In [24]:
import numpy as np
import pandas as pd
# Train Test Split
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

In [5]:
# ===============================
# Titanic ML Pipeline Full Code
# ===============================

import numpy as np
import pandas as pd

# Train Test Split
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

# Feature Selection
from sklearn.feature_selection import SelectKBest, chi2

# Model
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score

# Save Model
import pickle


# ===============================
# Load Dataset
# ===============================
df = pd.read_csv("train.csv")

# Show first rows
print(df.head())


# ===============================
# Drop Unnecessary Columns
# ===============================
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)


# ===============================
# Separate Features and Target
# ===============================
X = df.drop(columns=['Survived'])
y = df['Survived']


# ===============================
# Train Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ===============================
# Transformer 1: Missing Values
# ===============================
trf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(strategy='mean'), [2]),
    ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6])
], remainder='passthrough')


# ===============================
# Transformer 2: One Hot Encoding
# ===============================
trf2 = ColumnTransformer([
    ('ohe_sex_embarked',
     OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
     [1, 6])
], remainder='passthrough')


# ===============================
# Transformer 3: Scaling
# ===============================

trf3=ColumnTransformer([('scale',MinMaxScaler(),slice(0,10))])

## Transformer 4 : Featurre selection
trf4=SelectKBest(score_func=chi2,k=8)

#### ===============================
# Transformer 5: Model
# ===============================

trf5=DecisionTreeClassifier()

# ===============================
# Create Pipeline
# ===============================
pipe=Pipeline([('trf1',trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4),
    ('trf5', trf5)])
###########
####### TRAIN Model
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
print(accuracy_score(y_test,y_pred))

# ===============================
# Cross Validation Score
# ===============================
cv_score=cross_val_score(pipe,X_train,y=y_train,cv=5,scoring='accuracy').mean()
print("Cross validation :",cv_score)

### Grid Search CV
params={'trf5__max_depth':[1,2,3,4,5,None]}
grid=GridSearchCV(pipe,param_grid=params,scoring='accuracy',cv=5)
grid.fit(X_train,y_train)
print("Grid score")
print(grid.best_score_)

#### SAVE Pipeline
#pickle.dump(pipe,open('pipe.pkl','wb'))
#print('pipeline saved sucessfully')






   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
0.